In [9]:
import pandas as pd
import numpy as np

# Load Dataset and Create Mapping

In [12]:
drug_mappings = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 4 - Drug mappings.xlsx")
drug_mappings = drug_mappings[drug_mappings["RXNORM_CODE/RXNORM_EXTENSION_CODE (OHDSI)"] != 0]
drug_mappings["DRUG_CONCEPT_NAME"] = drug_mappings["DRUG_CONCEPT_NAME"].str.lower()

emb_desc = pd.read_csv("/data/giobbi/embeddings/DESC_GPT.csv", sep="\t")
emb_desc = emb_desc[["Drug Name", "Drug ID"]]
emb_desc["Drug Name"] = emb_desc["Drug Name"].str.lower()

In [13]:
merged_mappings = drug_mappings.merge(
    emb_desc[["Drug Name", "Drug ID"]], left_on="DRUG_CONCEPT_NAME", right_on="Drug Name", how="left"
)

In [14]:
# 1. Get unmatched rows from the first merge
unmatched = merged_mappings[merged_mappings["Drug ID"].isna()].copy()

# 2. Prepare the column for matching
unmatched.drop(columns=["Drug Name", "Drug ID"], inplace=True)
unmatched["DRUG_INITIAL_NAME"] = unmatched["DRUG_INITIAL_NAME"].str.lower()

# 3. Merge again using DRUG_INITIAL_NAME
second_merge = unmatched.merge(
    emb_desc[["Drug Name", "Drug ID"]], left_on="DRUG_INITIAL_NAME", right_on="Drug Name", how="left"
)

# 4. Optionally, concatenate the results back to the original merged_mappings
# (excluding the previously unmatched rows)
final_merged = pd.concat([merged_mappings[~merged_mappings["Drug ID"].isna()], second_merge], ignore_index=True)

In [15]:
#with pd.ExcelWriter("drug_mappings_and_emb_desc.xlsx") as writer:
#    final_merged.to_excel(writer, sheet_name="MergedMappings", index=False)
#    emb_desc.to_excel(writer, sheet_name="EmbDesc", index=False)

In [16]:
#final_merged.to_csv("mapping_rxnorm_drugbank.csv", index=False)

In [17]:
final_merged[final_merged["Drug ID"].isna()]

,DRUG_INITIAL_NAME,DRUG_CONCEPT_NAME,RXNORM_CODE/RXNORM_EXTENSION_CODE (OHDSI),SOURCE,Drug Name,Drug ID
5158,epoetin alfa,epoetin alfa,1301125,BNF_DDI,NaN,NaN
5159,epoetin beta,epoetin beta,19001311,BNF_DDI,NaN,NaN
5160,epoetin zeta,epoetin zeta,36878697,BNF_DDI,NaN,NaN
5164,cannabis extract,cannabis sativa whole extract,1559806,BNF_DDI,NaN,NaN
5166,trientine,trientine,19004969,BNF_DDI,NaN,NaN
...,...,...,...,...,...,...
5906,anti-d (rh0) immunoglobulin,rho(d) immune globulin,535714,BNF_Single,NaN,NaN
5909,follitropin alfa,follitropin alfa,1542948,BNF_Single,NaN,NaN
5910,lithium citrate,lithium citrate,767410,BNF_Single,NaN,NaN
5911,follitropin delta,follicle stimulating hormone,1588712,BNF_Single,NaN,NaN


# Create Graph Data

In [18]:
pos_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 1 - Positive Controls.xlsx")
pos_df["DRUG_1_CONCEPT_NAME"] = pos_df["DRUG_1_CONCEPT_NAME"].str.lower()
pos_df["DRUG_2_CONCEPT_NAME"] = pos_df["DRUG_2_CONCEPT_NAME"].str.lower()
pos_df = pos_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
pos_df.loc[:, "label"] = 1


neg_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 2 - Negative Controls.xlsx")
neg_df["DRUG_1_CONCEPT_NAME"] = neg_df["DRUG_1_CONCEPT_NAME"].str.lower()
neg_df["DRUG_2_CONCEPT_NAME"] = neg_df["DRUG_2_CONCEPT_NAME"].str.lower()
neg_df = neg_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
neg_df.loc[:, "label"] = 0

edges = pd.concat([pos_df, neg_df], ignore_index=True)

In [19]:
name_to_id = final_merged.set_index("DRUG_CONCEPT_NAME")["Drug ID"].to_dict()  # ['Drug ID'].to_dict()

edges["Drug1"] = edges["DRUG_1_CONCEPT_NAME"].map(name_to_id)
edges["Drug2"] = edges["DRUG_2_CONCEPT_NAME"].map(name_to_id)

In [20]:
edges

,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,label,Drug1,Drug2
0,bendroflumethiazide,desmopressin,1,DB00436,DB00035
1,chlorthalidone,desmopressin,1,DB00310,DB00035
2,desmopressin,clopamide,1,DB00035,DB13792
3,escitalopram,desmopressin,1,DB01175,DB00035
4,paroxetine,desmopressin,1,DB00715,DB00035
...,...,...,...,...,...
14825,fluticasone,clonidine,0,DB13867,DB00575
14826,fluticasone,perindopril,0,DB13867,DB00790
14827,bortezomib,ceftriaxone,0,DB00188,DB01212
14828,olanzapine,meloxicam,0,DB00334,DB00814


In [21]:
edges = edges.dropna(subset=["Drug1", "Drug2"]).copy()

In [22]:
import numpy as np


# Remove duplicate edges considering undirected pairs (A-B is same as B-A)
# Sort the drug IDs row-wise to normalize the order
sorted_edges = np.sort(edges[["Drug1", "Drug2"]].values, axis=1)

# Create a DataFrame from the sorted edges with the SAME INDEX as the original edges
sorted_edges_df = pd.DataFrame(sorted_edges, columns=["Drug1", "Drug2"], index=edges.index)

# Keep only rows that are not duplicates in the sorted version
edges = edges[~sorted_edges_df.duplicated()]

In [23]:
pos_edges = edges[edges["label"] == 1]
pos_edges = pos_edges[~(pos_edges["Drug1"].isna() | pos_edges["Drug2"].isna())]
GRAPH_URL = "/data/giobbi/CRESCENDDI/positive_edges_CRESCENDDI.csv"
#pos_edges[["Drug1", "Drug2"]].to_csv(GRAPH_URL, index=False, sep="\t")

In [24]:
edges = edges[~(edges["Drug1"].isna() | edges["Drug2"].isna())]
GRAPH_URL = "/data/giobbi/CRESCENDDI/CRESCENDDI.csv"
#edges[["Drug1", "Drug2", "label"]].to_csv(GRAPH_URL, index=False, sep="\t")

In [25]:
edges[(edges["Drug1"].isna() | edges["Drug2"].isna())]

,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,label,Drug1,Drug2


# Create combined dataset

In [27]:
cresc = pd.read_csv("/data/giobbi/CRESCENDDI/CRESCENDDI.csv", sep="\t")
#drugbank = pd.read_csv("https://raw.githubusercontent.com/liiniix/BioSNAP/master/ChCh-Miner/ChCh-Miner_durgbank-chem-chem.tsv", sep="\t")
#drugbank.to_csv("/data/giobbi/DRUGBANK/drugbank_graph.csv", index=False, sep="\t")
drugbank = pd.read_csv("/data/giobbi/GRAPH/drugbank_graph.csv", sep="\t")


In [28]:
drugbank

,Drug1,Drug2
0,DB00862,DB00966
1,DB00575,DB00806
2,DB01242,DB08893
3,DB01151,DB08883
4,DB01235,DB01275
...,...,...
48509,DB00542,DB01354
48510,DB00476,DB01239
48511,DB00621,DB01120
48512,DB00808,DB01356


## Anaysis of missing drug ids

In [31]:
edges

,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,label,Drug1,Drug2
0,bendroflumethiazide,desmopressin,1,DB00436,DB00035
1,chlorthalidone,desmopressin,1,DB00310,DB00035
2,desmopressin,clopamide,1,DB00035,DB13792
3,escitalopram,desmopressin,1,DB01175,DB00035
4,paroxetine,desmopressin,1,DB00715,DB00035
...,...,...,...,...,...
14824,paroxetine,fosinopril,0,DB00715,DB00492
14825,fluticasone,clonidine,0,DB13867,DB00575
14826,fluticasone,perindopril,0,DB13867,DB00790
14827,bortezomib,ceftriaxone,0,DB00188,DB01212


In [32]:
edges[edges["Drug1"].isna() | edges["Drug2"].isna()]

,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,label,Drug1,Drug2


In [33]:
# get missing drug ids from edges with drug names and occurence count
missing_drug_1 = edges[edges["DRUG_1_ID"].isna()]["DRUG_1_CONCEPT_NAME"].value_counts()
missing_drug_2 = edges[edges["DRUG_2_ID"].isna()]["DRUG_2_CONCEPT_NAME"].value_counts()

# stack df and group by index to sum counts as we don't care if missing in drug 1 or drug 2, sort descending
pd.concat([missing_drug_1, missing_drug_2], axis=1, keys=["Missing_DRUG_1_Count", "Missing_DRUG_2_Count"]).fillna(
    0
).astype(int).sum(axis=1).sort_values(ascending=False)

KeyError: 'DRUG_1_ID'

# Analyze common edges

In [34]:
AVAILABLE_GRAPHS = {
    "ChCh_Miner": "https://raw.githubusercontent.com/liiniix/BioSNAP/master/ChCh-Miner/ChCh-Miner_durgbank-chem-chem.tsv",
    "positive_edges_CRESCENDDI": "/data/giobbi/CRESCENDDI/positive_edges_CRESCENDDI.csv",
    "CRESCENDDI": "/data/giobbi/CRESCENDDI/CRESCENDDI.csv",
}
df_crescenddi = pd.read_csv(
    AVAILABLE_GRAPHS["CRESCENDDI"],
    sep="\t",
).rename(columns={"Drug1": "src", "Drug2": "dst"})
df_ChCh_Miner = pd.read_csv(
    AVAILABLE_GRAPHS["ChCh_Miner"],
    sep="\t",
).rename(columns={"Drug1": "src", "Drug2": "dst"})

In [35]:
df_crescenddi[df_crescenddi["label"] == 1]

,src,dst,label
0,DB00436,DB00035,1
1,DB00310,DB00035,1
2,DB00035,DB13792,1
3,DB01175,DB00035,1
4,DB00715,DB00035,1
...,...,...,...
4829,DB00281,DB01203,1
4830,DB00281,DB00501,1
4831,DB00501,DB01115,1
4832,DB00501,DB00608,1


In [36]:
df_ChCh_Miner

,src,dst
0,DB00862,DB00966
1,DB00575,DB00806
2,DB01242,DB08893
3,DB01151,DB08883
4,DB01235,DB01275
...,...,...
48509,DB00542,DB01354
48510,DB00476,DB01239
48511,DB00621,DB01120
48512,DB00808,DB01356


In [37]:
df_crescenddi_pos = df_crescenddi[df_crescenddi["label"] == 1][["src", "dst"]]

In [38]:
# get common edges between df_crescenddi and df_ChCh_Miner after making bidirectional
df_crescenddi_bi = pd.concat(
    [df_crescenddi_pos, df_crescenddi_pos.rename(columns={"src": "dst", "dst": "src"})], ignore_index=True
)
df_ChCh_Miner_bi = pd.concat(
    [df_ChCh_Miner, df_ChCh_Miner.rename(columns={"src": "dst", "dst": "src"})], ignore_index=True
)
common_edges = pd.merge(
    df_crescenddi_bi, df_ChCh_Miner_bi, on=["src", "dst"], how="inner"
)
print("Common edges:", common_edges.shape[0] // 2)  # divide by 2 to account for bidirectional duplication
# show edges that are only in df_crescenddi
only_in_crescenddi = pd.merge(
    df_crescenddi_bi, df_ChCh_Miner_bi, on=["src", "dst"], how="left", indicator=True
).query('_merge == "left_only"').drop(columns=["_merge"])
print("Only in CRESCENDDI:", only_in_crescenddi.shape[0] // 2)  # divide by 2 to account for bidirectional duplication
# show edges that are only in df_ChCh_Miner
only_in_ChCh_Miner = pd.merge(
    df_ChCh_Miner_bi, df_crescenddi_bi, on=["src", "dst"], how="left", indicator=True
).query('_merge == "left_only"').drop(columns=["_merge"])
print("Only in ChCh_Miner:", only_in_ChCh_Miner.shape[0] // 2)  # divide by 2 to account for bidirectional duplication

Common edges: 2067
Only in CRESCENDDI: 2767
Only in ChCh_Miner: 46447


# Further overlap analysis

## Analyze CRESCENDDI only

In [39]:
drug_mappings = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 4 - Drug mappings.xlsx")

In [40]:
drug_mappings

,DRUG_INITIAL_NAME,DRUG_CONCEPT_NAME,RXNORM_CODE/RXNORM_EXTENSION_CODE (OHDSI),SOURCE
0,Bortezomib,bortezomib,1336825,BNF_DDI
1,Nicardipine,nicardipine,1318137,BNF_DDI
2,Doxorubicin,doxorubicin,1338512,BNF_DDI
3,Thiopental,thiopental,700253,BNF_DDI
4,Bosentan,bosentan,1321636,BNF_DDI
...,...,...,...,...
6783,ganciclovir,ganciclovir,1757803,BNF_Single
6784,ganirelix,ganirelix,1536743,BNF_Single
6785,gefitinib,gefitinib,1319193,BNF_Single
6786,gelatin,gelatin,19058092,BNF_Single


In [41]:
# get number of nunique names in DRUG_CONCEPT_NAME
drugs_in_mapping = set(drug_mappings["DRUG_CONCEPT_NAME"].str.lower().unique())
len(drugs_in_mapping)

2289

In [139]:
pos_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 1 - Positive Controls.xlsx")
pos_df["DRUG_1_CONCEPT_NAME"] = pos_df["DRUG_1_CONCEPT_NAME"].str.lower()
pos_df["DRUG_2_CONCEPT_NAME"] = pos_df["DRUG_2_CONCEPT_NAME"].str.lower()
pos_df = pos_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
pos_df.loc[:, "label"] = 1


neg_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 2 - Negative Controls.xlsx")
neg_df["DRUG_1_CONCEPT_NAME"] = neg_df["DRUG_1_CONCEPT_NAME"].str.lower()
neg_df["DRUG_2_CONCEPT_NAME"] = neg_df["DRUG_2_CONCEPT_NAME"].str.lower()
neg_df = neg_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
neg_df.loc[:, "label"] = 0

edges = pd.concat([pos_df, neg_df], ignore_index=True)


In [144]:
# number of unique drugs in edges
nodes_in_graph = set(neg_df['DRUG_1_CONCEPT_NAME']) | set(neg_df['DRUG_2_CONCEPT_NAME'])
len(nodes_in_graph)

435

In [145]:
nodes_in_graph = set(neg_df['DRUG_1_CONCEPT_NAME']) | set(neg_df['DRUG_2_CONCEPT_NAME'])

In [146]:
len(nodes_in_graph)

435

In [147]:
len( nodes_in_graph & drugs_in_mapping )

435

------------

In [152]:
df_small = df_crescenddi #[df_crescenddi['label'] == 1]

edges_in_graph = set(df_small['src']) | set(df_small['dst'])
len(set(edges_in_graph))

444

In [137]:
df_small

,src,dst,label
9988,DB00436,DB00787,0
9989,DB00211,DB00479,0
9990,DB01118,DB00881,0
9991,DB00373,DB01268,0
9992,DB00842,DB00461,0
...,...,...,...
14411,DB13867,DB00575,0
14412,DB13867,DB00790,0
14413,DB00188,DB01212,0
14414,DB00334,DB00814,0


In [126]:
df_crescenddi[df_crescenddi["label"] == 1]

,src,dst,label
0,DB00436,DB00035,1
1,DB00310,DB00035,1
2,DB00035,DB13792,1
3,DB01175,DB00035,1
4,DB00715,DB00035,1
...,...,...,...
9983,DB00501,DB00608,1
9984,DB00501,DB01171,1
9985,DB00501,DB01171,1
9986,DB00501,DB01171,1


## Description

In [2]:
emb_desc = pd.read_csv("/data/giobbi/embeddings/Dr_Desc_GPT.csv", sep="\t")
emb_desc = emb_desc[["Drug Name", "Drug ID"]]
emb_desc["Drug Name"] = emb_desc["Drug Name"].str.lower()

NameError: name 'pd' is not defined

In [157]:
len(set(emb_desc['Drug Name']))

8722

### ChCh Miner

In [5]:
path = "https://raw.githubusercontent.com/liiniix/BioSNAP/master/ChCh-Miner/ChCh-Miner_durgbank-chem-chem.tsv"
miner = pd.read_csv(path, sep="\t")

In [6]:
miner

,Drug1,Drug2
0,DB00862,DB00966
1,DB00575,DB00806
2,DB01242,DB08893
3,DB01151,DB08883
4,DB01235,DB01275
...,...,...
48509,DB00542,DB01354
48510,DB00476,DB01239
48511,DB00621,DB01120
48512,DB00808,DB01356


In [7]:
# find unique drugs in miner
miner_drugs = set(miner['Drug1']) | set(miner['Drug2'])
len(miner_drugs)

1514

In [8]:
# miner filtered my emb_desc 
miner_filtered = miner[
    (miner['Drug1'].isin(emb_desc['Drug ID'])) & (miner['Drug2'].isin(emb_desc['Drug ID']))
]

NameError: name 'emb_desc' is not defined

In [167]:
miner_filtered

,Drug1,Drug2
0,DB00862,DB00966
1,DB00575,DB00806
2,DB01242,DB08893
3,DB01151,DB08883
4,DB01235,DB01275
...,...,...
48509,DB00542,DB01354
48510,DB00476,DB01239
48511,DB00621,DB01120
48512,DB00808,DB01356


In [168]:
unique_drugs_miner = set(miner_filtered['Drug1']) | set(miner_filtered['Drug2'])
len(unique_drugs_miner)

1491

## CRESCENDDI and MINER

In [179]:
cresc = df_crescenddi[df_crescenddi['label'] == 1]

In [180]:
# find overlap in unique drugs from crescenddi and miner
unique_drugs_miner = set(miner_filtered['Drug1']) | set(miner_filtered['Drug2'])
unique_drugs_crescenddi = set(cresc['src']) | set(cresc['dst'])
len( unique_drugs_miner & unique_drugs_crescenddi )

415

In [194]:
orig = df_crescenddi_pos.copy()
df_crescenddi_pos = df_crescenddi[df_crescenddi["label"] == 0][["src", "dst"]]

In [195]:
# get common edges between df_crescenddi and df_ChCh_Miner after making bidirectional
df_crescenddi_bi = pd.concat(
    [df_crescenddi_pos, df_crescenddi_pos.rename(columns={"src": "dst", "dst": "src"})], ignore_index=True
)
df_ChCh_Miner_bi = pd.concat(
    [df_ChCh_Miner, df_ChCh_Miner.rename(columns={"src": "dst", "dst": "src"})], ignore_index=True
)
common_edges = pd.merge(
    df_crescenddi_bi, df_ChCh_Miner_bi, on=["src", "dst"], how="inner"
)
print("Common edges:", common_edges.shape[0] // 2)  # divide by 2 to account for bidirectional duplication
# show edges that are only in df_crescenddi
only_in_crescenddi = pd.merge(
    df_crescenddi_bi, df_ChCh_Miner_bi, on=["src", "dst"], how="left", indicator=True
).query('_merge == "left_only"').drop(columns=["_merge"])
print("Only in CRESCENDDI:", only_in_crescenddi.shape[0] // 2)  # divide by 2 to account for bidirectional duplication
# show edges that are only in df_ChCh_Miner
only_in_ChCh_Miner = pd.merge(
    df_ChCh_Miner_bi, df_crescenddi_bi, on=["src", "dst"], how="left", indicator=True
).query('_merge == "left_only"').drop(columns=["_merge"])
print("Only in ChCh_Miner:", only_in_ChCh_Miner.shape[0] // 2)  # divide by 2 to account for bidirectional duplication

Common edges: 194
Only in CRESCENDDI: 4234
Only in ChCh_Miner: 48341
